# Fix Winners producers

With the initial scrap, we do not necessarily get the good details for the producers and artist columns in the file. This file aims to reconcile the data correctly

In [108]:
import pandas
import pathlib

In [109]:
YEAR = 1970

In [110]:
path = pathlib.Path('.').joinpath('tmp/producers_to_match.json').absolute()

In [111]:
producers_to_match = pandas.read_json(path)

In [112]:
df = pandas.read_csv('grammy_winners_v2.csv')

In [113]:
partial_df = df.query('year == @YEAR')

In [114]:
partial_df[['id', 'year', 'producers', 'winner']].head(n=3)

,id,year,producers,winner
2761,2035,1970,NaN,False
2762,2036,1970,NaN,False
2763,2037,1970,Let It Be,True


In [115]:
values_to_complete = partial_df[(partial_df.winner == True) & partial_df.producers.isna()][['id', 'category', 'song_or_album', 'artist', 'producers']]

In [116]:
values_to_complete.head(n=2)

,id,category,song_or_album,artist,producers
2773,2047,Best Recording For Children,Sesame Street,(The Muppets),NaN
2778,2052,Best Comedy Recording,The Devil Made Me Buy This Dress,Flip Wilson,NaN


In [139]:
# completed = []

# for item in values_to_complete.itertuples():
#     matched = df.loc[
#         lambda x: x.song_or_album == df.loc[item.Index, 'song_or_album'],
#         'category'
#     ]
#     if not matched.empty:
#         df.loc[matched.index, 'producers'] = None
#         print(f"Matched producers for ID {item.id}: {item.producers}")
#         completed.append(item.id)
#         continue

empty_count = 0

for item in producers_to_match.itertuples():
    matched = values_to_complete.loc[
        lambda x: (
            x.song_or_album.str.lower() == item.title.lower()
        ),
        'producers'
    ]

    if not matched.empty:
        if item.producers is None or item.producers == '':
            empty_count += 1

        values_to_complete.loc[matched.index, 'producers'] = item.producers or pandas.NA
        print(f"Matched: {item.title} ({item.award}): {item.producers}")
        continue

Matched: Bridge Over Troubled Water (Record Of The Year): Roy Halee & Simon And Garfunkel (Art Garfunkel & Paul Simon), producers
Matched: Bridge Over Troubled Water (Album Of The Year): Roy Halee & Simon And Garfunkel (Art Garfunkel & Paul Simon), producers
Matched: Bridge Over Troubled Water (Song Of The Year): Paul Simon, songwriter (Simon And Garfunkel)
Matched: Bridge Over Troubled Water (Best Arrangement Accompanying Vocalist(s)): Ernie Freeman, Art Garfunkel, Jimmie Haskell, Larry Knechtel & Paul Simon, arrangers (Simon And Garfunkel)
Matched: Bridge Over Troubled Water (Best Engineered Recording - Non-Classical): Roy Halee, engineer (Simon And Garfunkel)
Matched: I'll Never Fall In Love Again (Best Contemporary Vocal Performance, Female): None
Matched: Don't Play That Song (Best R&B Vocal Performance, Female): None
Matched: The Thrill Is Gone (Best R&B Vocal Performance, Male): None
Matched: Didn't I (Blow Your Mind This Time) (Best R&B Performance By A Duo Or Group, Vocal Or I

In [129]:
print(f'Values with empty producers: {empty_count}')

Values with empty producers: 17


In [130]:
values_to_complete.head(n=5)

,id,category,song_or_album,artist,producers
2773,2047,Best Recording For Children,Sesame Street,(The Muppets),<NA>
2778,2052,Best Comedy Recording,The Devil Made Me Buy This Dress,Flip Wilson,<NA>
2783,2057,Best Spoken Word Recording,Why I Oppose The War In Vietnam,Martin Luther King Jr.,<NA>
2789,2063,Best Jazz Performance - Small Group Or Soloist...,Alone,Bill Evans,<NA>
2797,2070,Best Jazz Performance - Large Group Or Soloist...,Bitches Brew,Miles Davis,<NA>


In [131]:
for item in values_to_complete.itertuples(name='UpdatedItem'):
    df.loc[lambda x: x.id == item.id, 'producers'] = item.producers

In [132]:
df.to_csv('grammy_winners_v2.csv', index=False)

In [354]:
missed_producers = df[df.producers == df.song_or_album][['url', 'year', 'producers', 'song_or_album', 'artist']]

In [355]:
TMP_DIR = pathlib.Path(__name__).parent.joinpath('tmp').absolute()

In [356]:
missed_producers.to_csv(TMP_DIR / 'missed_producers.csv', index_label='id')